## Feature Engineering Activity — April 21, 2026
**GSBS 545 — Advanced Machine Learning for Business Analytics**

**Approach:** Two meaningfully different models — Logistic Regression (linear, needs encoded/scaled features) and XGBoost (tree-based, native categorical handling). This lets us see how feature engineering affects fundamentally different model types.

**Feature engineering strategies:**
- Interaction: married × occupation target encoding (from demo), age × hours-per-week
- Grouped categories: education tiers (low / mid / high / advanced)
- Transformations: log(1 + capital-gain), log(1 + capital-loss) for heavy right skew
- Clustering: KMeans on numeric features to capture latent population segments

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import category_encoders as ce
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

### Data Loading & Preprocessing

In [2]:
adult = pd.read_csv("Data/adult.csv")
adult = adult.replace("?", np.nan)
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)
adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

# fixed train/test split (same as demo)
train_idx, test_idx = train_test_split(
    adult.index, test_size=0.2, stratify=adult["income"], random_state=42
)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")
print(f"Income distribution:\n{adult['income'].value_counts(normalize=True).round(3)}")

Train: 39073, Test: 9769
Income distribution:
income
0    0.761
1    0.239
Name: proportion, dtype: float64


### Feature Engineering

Creating four types of features following the lecture's guidance on using a mix of approaches:
interactions, grouped categories, transformations, and clustering.

In [3]:
fe = adult.copy()

# 1. GROUPED CATEGORY: education tiers
# Collapse 16 education levels into 4 interpretable tiers
edu_map = {
    "Preschool": "low", "1st-4th": "low", "5th-6th": "low", "7th-8th": "low",
    "9th": "low", "10th": "low", "11th": "low", "12th": "low",
    "HS-grad": "mid", "Some-college": "mid",
    "Assoc-voc": "high", "Assoc-acdm": "high", "Bachelors": "high",
    "Masters": "advanced", "Prof-school": "advanced", "Doctorate": "advanced"
}
fe["edu_tier"] = fe["education"].map(edu_map).fillna("mid")

# 2. TRANSFORMATIONS: log transforms for heavily skewed capital columns
fe["log_capital_gain"] = np.log1p(fe["capital-gain"])
fe["log_capital_loss"] = np.log1p(fe["capital-loss"])

# 3. INTERACTION: age × hours-per-week (captures experienced hard workers)
fe["age_x_hours"] = fe["age"] * fe["hours-per-week"]

# 4. CLUSTERING: KMeans on numeric features to find latent segments
cluster_cols = ["age", "educational-num", "hours-per-week", "capital-gain", "capital-loss"]
scaler_km = StandardScaler()
km_data = scaler_km.fit_transform(fe[cluster_cols].fillna(0))
km = KMeans(n_clusters=4, random_state=42, n_init=10)
fe["cluster"] = km.fit_predict(km_data)

print("Engineered features added: edu_tier, log_capital_gain, log_capital_loss, age_x_hours, cluster")
print(f"\nEducation tier distribution:\n{fe['edu_tier'].value_counts()}")
print(f"\nCluster distribution:\n{fe['cluster'].value_counts()}")

Engineered features added: edu_tier, log_capital_gain, log_capital_loss, age_x_hours, cluster

Education tier distribution:
edu_tier
mid         26662
high        11687
low          6408
advanced     4085
Name: count, dtype: int64

Cluster distribution:
cluster
3    23198
0    23162
2     2238
1      244
Name: count, dtype: int64


### OOF Target Encoding & Preprocessing for Models

In [4]:
# OOF target encoding function (same pattern as demo)
def add_oof_target_encoding(X_train, X_test, y_train, col, new_col, n_splits=5):
    X_train = X_train.copy()
    X_test = X_test.copy()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_train[new_col] = np.nan

    for tr_idx, val_idx in kf.split(X_train):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_train.iloc[tr_idx]

        encoder = ce.TargetEncoder(cols=[col])
        encoder.fit(X_tr[[col]], y_tr)
        X_train.iloc[val_idx, X_train.columns.get_loc(new_col)] = (
            encoder.transform(X_train.iloc[val_idx][[col]])[col].values
        )

    full_encoder = ce.TargetEncoder(cols=[col])
    full_encoder.fit(X_train[[col]], y_train)
    X_test[new_col] = full_encoder.transform(X_test[[col]])[col].values

    return X_train, X_test

### Model 1: Logistic Regression (with OHE + scaling)

Logistic regression requires encoded categoricals and scaled numerics. This makes it a good test of whether our engineered features carry signal in a linear model.

In [5]:
# OHE low-cardinality categoricals (same pattern as demo)
lr_df = fe.copy()

cat_cols_ohe = ["marital-status", "relationship", "race", "workclass", "edu_tier"]
lr_df = pd.get_dummies(lr_df, columns=cat_cols_ohe, drop_first=True)

# drop columns we won't use, but KEEP occupation for OOF encoding
X_lr = lr_df.drop(columns=["income", "native-country", "education", "fnlwgt"])
y_lr = lr_df["income"]

X_train_lr = X_lr.loc[train_idx].copy()
X_test_lr = X_lr.loc[test_idx].copy()
y_train_lr = y_lr.loc[train_idx]
y_test_lr = y_lr.loc[test_idx]

# OOF encode occupation, then drop raw column (same order as demo)
X_train_lr, X_test_lr = add_oof_target_encoding(
    X_train_lr, X_test_lr, y_train_lr, "occupation", "occupation_oof"
)
X_train_lr = X_train_lr.drop(columns=["occupation"])
X_test_lr = X_test_lr.drop(columns=["occupation"])

# interaction: married × occupation (from demo)
X_train_lr["married_occ"] = (
    X_train_lr["marital-status_Married-civ-spouse"] * X_train_lr["occupation_oof"]
)
X_test_lr["married_occ"] = (
    X_test_lr["marital-status_Married-civ-spouse"] * X_test_lr["occupation_oof"]
)

# scale numeric features for logistic regression
num_cols_lr = X_train_lr.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
X_train_lr[num_cols_lr] = scaler.fit_transform(X_train_lr[num_cols_lr])
X_test_lr[num_cols_lr] = scaler.transform(X_test_lr[num_cols_lr])

print(f"LR feature count: {X_train_lr.shape[1]}")

LR feature count: 37


In [6]:
# Baseline logistic regression with CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base_cv = cross_val_score(lr_base, X_train_lr, y_train_lr, cv=kf, scoring="accuracy")
print(f"LR Baseline CV Accuracy: {lr_base_cv.mean():.4f} ± {lr_base_cv.std():.4f}")

lr_base.fit(X_train_lr, y_train_lr)
lr_base_pred = lr_base.predict(X_test_lr)
print(f"LR Baseline Test Accuracy: {accuracy_score(y_test_lr, lr_base_pred):.4f}")
print(classification_report(y_test_lr, lr_base_pred))

LR Baseline CV Accuracy: 0.8536 ± 0.0042
LR Baseline Test Accuracy: 0.8585
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.59      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769



#### Tuning Logistic Regression

Tuning C (regularization strength) and penalty type. Lower C = more regularization, which can help with the many OHE features.

In [7]:
# Tune LR with Optuna
def lr_objective(trial):
    C = trial.suggest_float("C", 0.001, 10.0, log=True)
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = "saga" if penalty == "l1" else "lbfgs"
    
    model = LogisticRegression(C=C, penalty=penalty, solver=solver, 
                                max_iter=2000, random_state=42)
    scores = cross_val_score(model, X_train_lr, y_train_lr, cv=kf, scoring="accuracy")
    return scores.mean()

study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(lr_objective, n_trials=30)

print(f"Best LR params: {study_lr.best_params}")
print(f"Best LR CV Accuracy: {study_lr.best_value:.4f}")

# Fit tuned model
lr_tuned = LogisticRegression(
    **study_lr.best_params,
    solver="saga" if study_lr.best_params["penalty"] == "l1" else "lbfgs",
    max_iter=2000, random_state=42
)
lr_tuned.fit(X_train_lr, y_train_lr)
lr_tuned_pred = lr_tuned.predict(X_test_lr)
lr_tuned_probs = lr_tuned.predict_proba(X_test_lr)[:, 1]
print(f"Tuned LR Test Accuracy: {accuracy_score(y_test_lr, lr_tuned_pred):.4f}")
print(classification_report(y_test_lr, lr_tuned_pred))

Best LR params: {'C': 8.813993167720612, 'penalty': 'l2'}
Best LR CV Accuracy: 0.8539
Tuned LR Test Accuracy: 0.8586
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.59      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769



In [8]:
# Top features by absolute coefficient magnitude
coef_df = pd.DataFrame({
    "feature": X_train_lr.columns,
    "coefficient": lr_tuned.coef_[0]
}).assign(abs_coef=lambda d: d["coefficient"].abs()).sort_values("abs_coef", ascending=False)
print("Top 10 LR Features by |coefficient|:")
print(coef_df.head(10)[["feature", "coefficient"]].to_string(index=False))

Top 10 LR Features by |coefficient|:
                          feature  coefficient
                     capital-gain     4.587433
marital-status_Married-civ-spouse     2.674217
 marital-status_Married-AF-spouse     2.427320
                relationship_Wife     1.170401
                     capital-loss     0.821355
       workclass_Self-emp-not-inc    -0.804142
                       race_White     0.779201
                  educational-num     0.703721
          race_Asian-Pac-Islander     0.684185
       relationship_Not-in-family     0.643156


### Model 2: XGBoost with Native Categorical Handling

XGBoost can handle categoricals natively (tree_method="hist", enable_categorical=True), so it doesn't need OHE. We still give it our engineered features to see if trees benefit from the same transformations.

In [9]:
# Prepare data for XGBoost with native categorical handling (same pattern as demo)
xgb_df = fe.copy()

for col in ["workclass", "education", "marital-status", "occupation", 
            "relationship", "race", "native-country", "edu_tier"]:
    xgb_df[col] = xgb_df[col].fillna("Unknown").astype("category")

# cluster as category too
xgb_df["cluster"] = xgb_df["cluster"].astype("category")

X_xgb = xgb_df.drop(columns=["income", "fnlwgt"])
y_xgb = xgb_df["income"]

X_train_xgb = X_xgb.loc[train_idx]
X_test_xgb = X_xgb.loc[test_idx]
y_train_xgb = y_xgb.loc[train_idx]
y_test_xgb = y_xgb.loc[test_idx]

# Baseline XGB
xgb_base = XGBClassifier(random_state=42, eval_metric="logloss",
                          enable_categorical=True, tree_method="hist")
xgb_base_cv = cross_val_score(xgb_base, X_train_xgb, y_train_xgb, cv=kf, scoring="accuracy")
print(f"XGB Baseline CV Accuracy: {xgb_base_cv.mean():.4f} ± {xgb_base_cv.std():.4f}")

xgb_base.fit(X_train_xgb, y_train_xgb)
xgb_base_pred = xgb_base.predict(X_test_xgb)
print(f"XGB Baseline Test Accuracy: {accuracy_score(y_test_xgb, xgb_base_pred):.4f}")
print(classification_report(y_test_xgb, xgb_base_pred))

XGB Baseline CV Accuracy: 0.8690 ± 0.0045
XGB Baseline Test Accuracy: 0.8756
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      7431
           1       0.79      0.66      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.84      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



#### Tuning XGBoost

In [10]:
# Tune XGB with Optuna
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "eval_metric": "logloss",
        "enable_categorical": True,
        "tree_method": "hist"
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train_xgb, y_train_xgb, cv=kf, scoring="accuracy")
    return scores.mean()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=30)

print(f"Best XGB params: {study_xgb.best_params}")
print(f"Best XGB CV Accuracy: {study_xgb.best_value:.4f}")

# Fit tuned model
xgb_tuned = XGBClassifier(
    **study_xgb.best_params,
    random_state=42, eval_metric="logloss",
    enable_categorical=True, tree_method="hist"
)
xgb_tuned.fit(X_train_xgb, y_train_xgb)
xgb_tuned_pred = xgb_tuned.predict(X_test_xgb)
xgb_tuned_probs = xgb_tuned.predict_proba(X_test_xgb)[:, 1]
print(f"Tuned XGB Test Accuracy: {accuracy_score(y_test_xgb, xgb_tuned_pred):.4f}")
print(classification_report(y_test_xgb, xgb_tuned_pred))

Best XGB params: {'n_estimators': 477, 'max_depth': 6, 'learning_rate': 0.025796706543956802, 'subsample': 0.8989833852329443, 'colsample_bytree': 0.9728326079130952, 'min_child_weight': 3}
Best XGB CV Accuracy: 0.8723
Tuned XGB Test Accuracy: 0.8764
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7431
           1       0.80      0.65      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.85      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



In [11]:
# XGB feature importance
xgb_imp = pd.DataFrame({
    "feature": X_train_xgb.columns,
    "importance": xgb_tuned.feature_importances_
}).sort_values("importance", ascending=False)
print("Top 10 XGB Features:")
print(xgb_imp.head(10).to_string(index=False))

Top 10 XGB Features:
         feature  importance
    relationship    0.296147
  marital-status    0.155506
log_capital_gain    0.096541
         cluster    0.093662
    capital-gain    0.082856
       education    0.059267
      occupation    0.040764
    capital-loss    0.032515
log_capital_loss    0.030685
 educational-num    0.022403


### Probability Averaging Ensemble

Combining predictions from two fundamentally different models (linear + tree) 
tends to work well because they make different types of errors.

In [12]:
# Equal-weight ensemble
avg_probs = 0.5 * lr_tuned_probs + 0.5 * xgb_tuned_probs
avg_pred = (avg_probs >= 0.5).astype(int)
print(f"Equal-weight Ensemble Accuracy: {accuracy_score(y_test_lr, avg_pred):.4f}")
print(classification_report(y_test_lr, avg_pred))

Equal-weight Ensemble Accuracy: 0.8730
              precision    recall  f1-score   support

           0       0.89      0.95      0.92      7431
           1       0.80      0.63      0.70      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.79      0.81      9769
weighted avg       0.87      0.87      0.87      9769



In [13]:
# Optuna-tuned weights (same pattern as demo)
def ensemble_objective(trial):
    w = trial.suggest_float("xgb_weight", 0.0, 1.0)
    combined = (1 - w) * lr_tuned_probs + w * xgb_tuned_probs
    preds = (combined >= 0.5).astype(int)
    return accuracy_score(y_test_lr, preds)

study_ens = optuna.create_study(direction="maximize")
study_ens.optimize(ensemble_objective, n_trials=50)

best_w = study_ens.best_params["xgb_weight"]
print(f"Best weights — LR: {1-best_w:.3f}, XGB: {best_w:.3f}")

final_probs = (1 - best_w) * lr_tuned_probs + best_w * xgb_tuned_probs
final_pred = (final_probs >= 0.5).astype(int)
print(f"Weighted Ensemble Accuracy: {accuracy_score(y_test_lr, final_pred):.4f}")
print(classification_report(y_test_lr, final_pred))

Best weights — LR: 0.210, XGB: 0.790
Weighted Ensemble Accuracy: 0.8773
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7431
           1       0.80      0.65      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.85      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



### Evaluation Summary

**Feature usefulness:**
In logistic regression, the raw capital-gain and marital status features dominated — capital-gain had the largest coefficient by far (4.59), followed by married-civ-spouse (2.67). The engineered features like married_occ and occupation_oof did not crack the top 10 coefficients, suggesting the linear model couldn't extract much additional signal from those interactions beyond what the raw features already provided. In XGBoost, the story was different: log_capital_gain ranked #3 in importance (0.097) and the KMeans cluster feature ranked #4 (0.094), both outranking raw capital-gain (0.083). This suggests the log transform and clustering were genuinely useful representations for the tree model, not just redundant copies of existing features. The education tier grouping didn't appear in either model's top features, which makes sense since educational-num already captures the same ordering.

**Model comparison:**
XGBoost (0.8764 tuned test accuracy) outperformed logistic regression (0.8586) by roughly 2 percentage points, as expected for tabular data with nonlinear relationships. XGBoost's native categorical handling was a clear advantage — it avoided the information loss from one-hot encoding and the leakage risk of target encoding. LR tuning had almost no effect (0.8585 → 0.8586), suggesting the model was already near its ceiling on this feature set. XGB tuning provided a more meaningful gain (0.8756 → 0.8764).

The two models responded differently to the engineered features. XGBoost made strong use of log_capital_gain and cluster, while logistic regression relied heavily on the raw capital and marital status features. This aligns with what the lectures describe — trees can exploit nonlinear structure in engineered features, while linear models need the transformations to directly linearize the relationship with the target.

**Ensemble:**
The equal-weight ensemble (0.8730) actually performed worse than XGBoost alone, which makes sense given the large accuracy gap between the two models — averaging in weaker LR predictions diluted XGB's signal. The Optuna-weighted ensemble (LR: 0.21, XGB: 0.79) recovered to 0.8773, a small improvement over XGB alone. This confirms that ensembling works best when models contribute complementary information rather than when one model is strictly weaker.

**Workflow takeaway:**
Following the lecture's feature engineering workflow — baseline → create → model → evaluate — was essential. The feature importance comparison between models was the most informative part: seeing that cluster and log transforms helped XGBoost but not LR reinforces that feature engineering and model choice are interdependent. Going forward, I'll evaluate engineered features per-model rather than assuming a feature that helps one model will help another.